# Price differences before and after energy crisis

## Set up

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

from scripts.utils import scale_font_latex
from scripts.countries import country_codes
from scripts.constants import ENERGY_CRISIS, START, END, years
from scripts.config import paths

In [2]:
COUNTRY_CODE = "FR"


data = pd.read_csv(
    paths["data_full"],
    parse_dates=["timestamp"],
).set_index("timestamp")
data.index = pd.to_datetime(data.index, utc="utc")


# European prices da
price = pd.read_csv(paths["european_prices_da"], parse_dates=["timestamp"]).set_index(
    "timestamp"
)
price = price.truncate(before=START, after=END)
zones = [cc for cc in country_codes if cc not in ["IT_ROSN", "IT_CALA"]]

## Histogramm of price difference before and after energy crisis

In [ ]:
def plot_price_diff(price_1, price_2, color, ax, label):
    """
    Plot the price difference (price_1 - price_2) between two price series over time.

    Parameters:
    price_1 (pandas.Series): The first price series.
    price_2 (pandas.Series): The second price series.
    color (str): The color for the plot.
    ax (matplotlib.axes.Axes): The axes on which to plot the price difference.
    label (str): The label for the plot legend.

    Returns:
    None
    """
    price_diff = pd.DataFrame(index=pd.date_range(START, END, freq="h"))
    price_diff.index.name = "timestamp"
    price_diff["price_diff"] = price_1 - price_2
    price_diff["rel_price_diff"] = 2 * (price_1 - price_2) / (price_1 + price_2)
    price_diff.replace(
        [np.inf, -np.inf], np.nan, inplace=True
    )  # Replace infinities with NaN
    price_diff.dropna(inplace=True)  # Drop rows with NaN values
    price_diff_std = price_diff.resample("1ME").std()["price_diff"]
    price_diff_mean = price_diff.resample("1ME").mean()["price_diff"]

    # Plot the monthly mean
    ax.plot(
        price_diff_mean.index,
        price_diff_mean,
        marker="o",
        label=label,
        color=color,
    )

    # Fill between the mean ± std
    ax.fill_between(
        price_diff_mean.index,
        price_diff_mean - price_diff_std,
        price_diff_mean + price_diff_std,
        color=color,
        alpha=0.3,
    )
    # Add a horizontal dashed line at y = 0
    ax.axhline(0, color="black", linewidth=1)
    # Add a vertical line at energy crisis
    ax.axvline(x=ENERGY_CRISIS, color="black", linestyle="dashed", linewidth=1)
    ax.legend(loc="upper left")
    ax.set_ylabel("Price diff. (EUR)")
    ax.set_xlim(START, END)


def plot_export_balance(export, color, ax, label):
    ax.set_ylabel("Export (MW)")
    ax.plot(export.resample("1ME").mean(), label=label, marker="o", color=color)
    ax.legend(loc="upper left")
    ax.fill_between(
        export.resample("1ME").mean().index,
        export.resample("1ME").mean() - export.resample("1ME").std(),
        export.resample("1ME").mean() + export.resample("1ME").std(),
        color=color,
        alpha=0.3,
    )
    # Add a vertical line at energy crisis
    ax.axvline(x=ENERGY_CRISIS, color="black", linestyle="dashed", linewidth=1)
    # Add a horizontal dashed line at y = 0
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlim(START, END)


scale_font_latex(1.5)
colors = sns.color_palette("colorblind")
fig, axes = plt.subplots(4, 1, figsize=(16, 18), sharex=True)
plot_price_diff(price["FR"], price["DE_LU"], color=colors[0], ax=axes[0], label="DE-LU")
plot_price_diff(price["FR"], price["BE"], color=colors[1], ax=axes[1], label="BE")
plot_price_diff(price["FR"], price["ES"], color=colors[2], ax=axes[2], label="ES")
plot_price_diff(
    price["FR"], price["IT_NORD"], color=colors[3], ax=axes[3], label="IT-North"
)
plt.xlabel("Time")
fig_dir = "../reports/figures/price_diff"
os.makedirs(fig_dir, exist_ok=True)
fig.savefig(fig_dir + f"/price_diff_FR.pdf", bbox_inches="tight")
plt.show()

colors = sns.color_palette("colorblind")
fig, axes = plt.subplots(4, 1, figsize=(16, 18), sharex=True)
plot_export_balance(data["FR->DE_LU"], color=colors[0], ax=axes[0], label="DE-LU")
plot_export_balance(data["FR->BE"], color=colors[1], ax=axes[1], label="BE")
plot_export_balance(data["FR->ES"], color=colors[2], ax=axes[2], label="ES")
plot_export_balance(data["FR->IT_NORD"], color=colors[3], ax=axes[3], label="IT-North")
plt.xlabel("Time")